场景
- 主智能体中调用 delete_file 工具需要审批， 调用 read_file 工具不需要审批
- 子智能体 file-manager 中调用 delete_file 和 read_file 都需要审批

重点：
- 子智能体的 interrupt_on 参数的配置可以独立于主智能体， 实现对不同层级的惊喜控制

In [ ]:
# 定义工具

from langchain.tools import tool


@tool
def delete_file(file_path: str) -> str:
    """删除文件(危险操作)

    Args:
        file_path (str): 文件路径

    Returns:
        str: 删除结果
    """
    return f"{file_path} 已删除"


@tool
def read_file(file_path: str) -> str:
    """读取文件

    Args:
        file_path (str): 文件路径

    Returns:
        str: 文件内容
    """
    return f"文件{file_path}的内容是： 锄禾日当午"

In [ ]:
# 创建智能体
from langgraph.checkpoint.memory import MemorySaver
from langchain.chat_models import init_chat_model
import os
from deepagents import create_deep_agent

checkpointer = MemorySaver()

model = init_chat_model(
    model="opeanai:deepseek-v4-pro", base_url="http://api.deepseek.com/v1", api_key=os.getenv("DEEPSEEK_API")
)

agent = create_deep_agent(
    model=model,
    tools=[read_file, delete_file],
    interrupt_on={"read_file": False, "delete_file": True},
    subagents=[
        {
            "name": "file-manager",
            "description": "专门负责文件管理的子智能体",
            "system_prompt": "你是一个文件管理助手， 可以谨慎地删除和阅读文件。",
            "tools": [read_file, delete_file],
            "interrupt_on": {"read_file": True, "delete_file": True},
        }
    ],
    checkpointer=checkpointer,
)

# 调用
config = {"configurable": {"thread_id": "test"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "请让 file-manager 子智能体帮我读取 text.txt 的内容"}]}, config=config
)

from deepagents.middleware import MemoryMiddleware

from langgraph.types import Command

while result.get("__interrupt__"):
    result = agent.invoke(Command(resume={"decisions": []}))

_IncompleteInputError: incomplete input (140512677.py, line 37)